In [ ]:
# Import necessary libraries
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset
from torchvision import datasets, transforms
from torchvision.utils import make_grid
import numpy as np
import cv2
import os

In [ ]:
# Define the paths to your datasets
# Update these paths to point to your actual dataset locations
radius_image_dataset_path = '/path/to/your/radius/dataset'  # Update this path to your dataset location
tibia_image_dataset_path = '/path/to/your/tibia/dataset'  # Update this path to your dataset location

# Step 1: Collect image paths and labels
image_paths = []
labels = []

def collect_image_paths(folder, label):
    for filename in os.listdir(folder):
        if filename.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp')):
            path = os.path.join(folder, filename)
            image_paths.append(path)
            labels.append(label)

collect_image_paths(radius_image_dataset_path, 0)
collect_image_paths(tibia_image_dataset_path, 1)

In [ ]:
# Create Class Module for the CNN Model
class CNNmodel(nn.Module):
  def __init__(self):
    super().__init__()
    self.conv1 = nn.Conv2d(1, 6, 3, 1)
    self.conv2 = nn.Conv2d(6, 16, 3 , 1)
    self.conv3 = nn.Conv2d(16, 80, 3, 1)

    # Fully Connected Layers
    self.fc1 = nn.Linear(80 * 53 * 53, 120)
    self.fc2 = nn.Linear(120, 84)
    self.fc3 = nn.Linear(84, 3)

  def forward(self, X):
    X = F.relu(self.conv1(X))
    X = F.max_pool2d(X, 2, 2) # Pooling Layer
    X = F.relu(self.conv2(X))
    X = F.relu(self.conv3(X))
    X = F.max_pool2d(X, 2, 2) # Pooling Layer

    # Flatten
    X = X.view(-1, 80 * 53 * 53)

    # Fully Connected Layers
    X = F.relu(self.fc1(X))
    X = F.relu(self.fc2(X))
    X = self.fc3(X)

    return F.log_softmax(X, dim = 1)

In [ ]:
# Set random seed for reproducibility
torch.manual_seed(42)
model = CNNmodel()

In [ ]:
# Define paths for the brain tumor datasets
data = []
labels = []

# Helper to load all images from a folder
def load_images_from_folder(folder, label):
    for filename in os.listdir(folder):
        img_path = os.path.join(folder, filename)
        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        if img is not None:
            img = cv2.resize(img, (224, 224))  # Resize to fixed shape
            data.append(img)
            labels.append(label)

# Load images
load_images_from_folder(radius_image_dataset_path, 0)
load_images_from_folder(tibia_image_dataset_path, 1)

In [ ]:
# Convert data and labels to numpy arrays
# Ensure data is in the correct shape for CNN input
X = np.array(data).reshape(-1, 1, 224, 224)  # Shape: [N, C, H, W] for CNN
y = np.array(labels)

# Train Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify = labels)

# Convert numpy arrays to PyTorch tensors
X_train = torch.FloatTensor(X_train)
X_test = torch.FloatTensor(X_test)
y_train = torch.LongTensor(y_train)
y_test = torch.LongTensor(y_test)

In [ ]:
# Create TensorDatasets and DataLoaders
train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False)

In [ ]:
# Loss Function Optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr = 0.01)

In [ ]:
# Training the Model
# Set the number of epochs and initialize a list to store losses
epochs = 100
losses = []

for i in range(epochs):
    model.train()
    running_loss = 0.0
    for batch_X, batch_y in train_loader:
        optimizer.zero_grad()
        y_pred = model(batch_X)
        loss = criterion(y_pred, batch_y)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    avg_loss = running_loss / len(train_loader)
    losses.append(avg_loss)

    if i % 5 == 0:
        print(f"Epoch: {i}, Loss: {avg_loss:.4f}")

In [ ]:
# Plotting the training loss over epochs
plt.plot(range(epochs), losses)
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.title("Training Loss over Time")
plt.show()

In [ ]:
# Evaluating the Model
# Set the model to evaluation mode and calculate accuracy and loss on the test set
with torch.no_grad():
    model.eval()
    correct = 0
    total = 0
    total_loss = 0.0  # Track total test loss

    for batch_X, batch_y in test_loader:
        y_eval = model(batch_X)
        loss = criterion(y_eval, batch_y)  # Compute loss
        total_loss += loss.item()

        predicted = torch.argmax(y_eval, dim=1)
        correct += (predicted == batch_y).sum().item()
        total += batch_y.size(0)

accuracy = correct / total
avg_test_loss = total_loss / len(test_loader)

print(f"Test Accuracy: {accuracy * 100:.2f}%")
print(f"Test Loss: {avg_test_loss:.4f}")